# ML-Based Trading Strategy: LightGBM & ElasticNet
## Predicting Returns and Volatility for Position Sizing

**Strategy Overview:**
- **ElasticNet** predicts forward returns (linear baseline)
- **LightGBM** predicts forward returns (nonlinear model) and realized volatility
- **Entry/Exit Rules:** Based on predicted return magnitude and direction
- **Position Sizing:** Inverse-volatility weighted, scaled by predicted return confidence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## 1. Load & Prepare Data

In [ ]:
# ============================================================
# DATA LOADING
# Option A: Load from CSV (if you've already run data_download.py)
# Option B: Generate synthetic data for demonstration
# ============================================================

USE_SYNTHETIC = True  # Set to False if you have real data CSV
CSV_PATH = 'aapl_intraday_data.csv'

if not USE_SYNTHETIC:
    df_raw = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df_raw)} rows from {CSV_PATH}")
else:
    # Generate realistic synthetic AAPL-like data
    np.random.seed(42)
    n_points = 50000  # ~200 trading days of minute data
    
    dates = pd.bdate_range(start='2023-01-03', periods=n_points // 390 + 1)
    datetimes = []
    for d in dates:
        for m in range(390):  # 390 minutes in a trading day
            datetimes.append(d + pd.Timedelta(hours=9, minutes=30+m))
            if len(datetimes) >= n_points:
                break
        if len(datetimes) >= n_points:
            break
    
    # Simulate price with drift + volatility clustering (GBM with stochastic vol)
    price = 150.0
    prices = [price]
    mu = 0.0001  # slight positive drift per minute
    base_sigma = 0.002
    vol = base_sigma
    
    for i in range(1, n_points):
        # GARCH-like vol clustering
        vol = 0.94 * vol + 0.06 * base_sigma * (1 + 2 * abs(np.random.randn()))
        ret = mu + vol * np.random.randn()
        price = price * (1 + ret)
        prices.append(price)
    
    prices = np.array(prices)
    highs = prices * (1 + np.abs(np.random.randn(n_points)) * 0.001)
    lows = prices * (1 - np.abs(np.random.randn(n_points)) * 0.001)
    opens = prices + np.random.randn(n_points) * 0.05
    volumes = np.random.lognormal(mean=10, sigma=1.5, size=n_points).astype(int)
    
    df_raw = pd.DataFrame({
        'Datetime': datetimes[:n_points],
        'Open': opens,
        'High': highs,
        'Low': lows,
        'Close': prices,
        'Volume': volumes
    })
    print(f"Generated {len(df_raw)} synthetic data points")

print(df_raw.head())
print(f"\nDate range: {df_raw['Datetime'].min()} to {df_raw['Datetime'].max()}")

In [ ]:
# ============================================================
# FEATURE ENGINEERING (using DataCleaner logic inline)
# ============================================================

df = df_raw.copy()
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.set_index('Datetime').sort_index()
df = df.dropna().drop_duplicates()

# --- Returns ---
for p in [1, 5, 10, 20]:
    df[f'Return_{p}'] = df['Close'].pct_change(p)
    df[f'LogReturn_{p}'] = np.log(df['Close'] / df['Close'].shift(p))

# --- Moving Averages ---
for w in [5, 10, 20, 50, 200]:
    df[f'SMA_{w}'] = df['Close'].rolling(w).mean()
    df[f'EMA_{w}'] = df['Close'].ewm(span=w, adjust=False).mean()

# --- Volatility ---
for w in [5, 10, 20]:
    df[f'Volatility_{w}'] = df['Return_1'].rolling(w).std()

# --- RSI ---
for period in [14]:
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

# --- MACD ---
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Histogram'] = df['MACD'] - df['MACD_Signal']

# --- Bollinger Bands ---
df['BB_Middle'] = df['Close'].rolling(20).mean()
bb_std = df['Close'].rolling(20).std()
df['BB_Upper'] = df['BB_Middle'] + 2 * bb_std
df['BB_Lower'] = df['BB_Middle'] - 2 * bb_std
df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
df['BB_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

# --- Price Features ---
df['Price_Change'] = df['Close'] - df['Open']
df['Price_Change_Pct'] = (df['Close'] - df['Open']) / df['Open']
df['HL_Spread'] = df['High'] - df['Low']
df['HL_Spread_Pct'] = df['HL_Spread'] / df['Close']
df['Body_Size_Pct'] = abs(df['Close'] - df['Open']) / df['Close']

# --- Volume Features ---
for w in [5, 10, 20]:
    df[f'Volume_MA_{w}'] = df['Volume'].rolling(w).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_MA_20']

# --- MA Crossover Signals ---
df['SMA_5_20_Cross'] = (df['SMA_5'] - df['SMA_20']) / df['Close']
df['SMA_10_50_Cross'] = (df['SMA_10'] - df['SMA_50']) / df['Close']
df['Price_SMA20_Ratio'] = df['Close'] / df['SMA_20'] - 1

# --- Target Variables ---
FORWARD_PERIOD = 10  # Predict 10-bar forward return & volatility

df['Target_Return'] = df['Close'].pct_change(FORWARD_PERIOD).shift(-FORWARD_PERIOD)
df['Target_Volatility'] = df['Return_1'].rolling(FORWARD_PERIOD).std().shift(-FORWARD_PERIOD)

# Drop NaN rows
df = df.dropna()
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date range: {df.index.min()} to {df.index.max()}")

In [ ]:
# ============================================================
# DEFINE FEATURES & TARGETS
# ============================================================

# Feature columns (exclude targets and raw OHLCV to avoid leakage)
exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume',
                'Target_Return', 'Target_Volatility']
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

X = df[feature_cols].values
y_return = df['Target_Return'].values
y_vol = df['Target_Volatility'].values

## 2. Train/Test Split (Time-Based)

In [ ]:
# ============================================================
# TIME-BASED TRAIN/TEST SPLIT (80/20)
# ============================================================

split_idx = int(len(df) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_ret_train, y_ret_test = y_return[:split_idx], y_return[split_idx:]
y_vol_train, y_vol_test = y_vol[:split_idx], y_vol[split_idx:]

dates_train = df.index[:split_idx]
dates_test = df.index[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples ({dates_train[0].date()} to {dates_train[-1].date()})")
print(f"Test set:     {X_test.shape[0]} samples ({dates_test[0].date()} to {dates_test[-1].date()})")

## 3. Model Training

### 3a. ElasticNet — Return Prediction (Linear Baseline)

In [ ]:
# ============================================================
# ELASTICNET: RETURN PREDICTION
# ============================================================

# Use TimeSeriesSplit for cross-validation
tscv = TimeSeriesSplit(n_splits=5)

enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95],
    alphas=np.logspace(-6, -1, 50),
    cv=tscv,
    max_iter=10000,
    random_state=42,
    n_jobs=-1
)

enet_cv.fit(X_train_scaled, y_ret_train)

print(f"Best alpha: {enet_cv.alpha_:.6f}")
print(f"Best l1_ratio: {enet_cv.l1_ratio_:.2f}")

# Predictions
enet_pred_train = enet_cv.predict(X_train_scaled)
enet_pred_test = enet_cv.predict(X_test_scaled)

# Metrics
print(f"\n--- ElasticNet Return Prediction ---")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_ret_train, enet_pred_train)):.6f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_ret_test, enet_pred_test)):.6f}")
print(f"Train R²:   {r2_score(y_ret_train, enet_pred_train):.4f}")
print(f"Test  R²:   {r2_score(y_ret_test, enet_pred_test):.4f}")
print(f"Test  MAE:  {mean_absolute_error(y_ret_test, enet_pred_test):.6f}")

# Feature importance (non-zero coefficients)
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': enet_cv.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\nTop 10 ElasticNet features:")
print(coef_df[coef_df['Coefficient'] != 0].head(10).to_string(index=False))

### 3b. LightGBM — Return Prediction

In [ ]:
# ============================================================
# LIGHTGBM: RETURN PREDICTION
# ============================================================

lgb_return_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.01,
    'max_depth': 6,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1
}

lgb_return_model = lgb.LGBMRegressor(**lgb_return_params)

# Use early stopping with a validation set from the end of training
val_split = int(len(X_train) * 0.85)
lgb_return_model.fit(
    X_train[:val_split], y_ret_train[:val_split],
    eval_set=[(X_train[val_split:], y_ret_train[val_split:])],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

# Predictions
lgb_ret_pred_train = lgb_return_model.predict(X_train)
lgb_ret_pred_test = lgb_return_model.predict(X_test)

print(f"--- LightGBM Return Prediction ---")
print(f"Best iteration: {lgb_return_model.best_iteration_}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_ret_train, lgb_ret_pred_train)):.6f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_ret_test, lgb_ret_pred_test)):.6f}")
print(f"Train R²:   {r2_score(y_ret_train, lgb_ret_pred_train):.4f}")
print(f"Test  R²:   {r2_score(y_ret_test, lgb_ret_pred_test):.4f}")
print(f"Test  MAE:  {mean_absolute_error(y_ret_test, lgb_ret_pred_test):.6f}")

### 3c. LightGBM — Volatility Prediction

In [ ]:
# ============================================================
# LIGHTGBM: VOLATILITY PREDICTION
# ============================================================

lgb_vol_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.01,
    'max_depth': 5,
    'num_leaves': 20,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1
}

lgb_vol_model = lgb.LGBMRegressor(**lgb_vol_params)

lgb_vol_model.fit(
    X_train[:val_split], y_vol_train[:val_split],
    eval_set=[(X_train[val_split:], y_vol_train[val_split:])],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

# Predictions
lgb_vol_pred_train = lgb_vol_model.predict(X_train)
lgb_vol_pred_test = lgb_vol_model.predict(X_test)

print(f"--- LightGBM Volatility Prediction ---")
print(f"Best iteration: {lgb_vol_model.best_iteration_}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_vol_train, lgb_vol_pred_train)):.6f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_vol_test, lgb_vol_pred_test)):.6f}")
print(f"Train R²:   {r2_score(y_vol_train, lgb_vol_pred_train):.4f}")
print(f"Test  R²:   {r2_score(y_vol_test, lgb_vol_pred_test):.4f}")
print(f"Test  MAE:  {mean_absolute_error(y_vol_test, lgb_vol_pred_test):.6f}")

## 4. Feature Importance Analysis

In [ ]:
# ============================================================
# FEATURE IMPORTANCE PLOTS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# ElasticNet coefficients
top_enet = coef_df[coef_df['Coefficient'] != 0].head(15)
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top_enet['Coefficient']]
axes[0].barh(range(len(top_enet)), top_enet['Coefficient'].values, color=colors)
axes[0].set_yticks(range(len(top_enet)))
axes[0].set_yticklabels(top_enet['Feature'].values, fontsize=9)
axes[0].set_title('ElasticNet Coefficients (Return)', fontweight='bold')
axes[0].set_xlabel('Coefficient Value')
axes[0].invert_yaxis()

# LightGBM Return importance
ret_imp = pd.Series(lgb_return_model.feature_importances_, index=feature_cols)
ret_imp = ret_imp.nlargest(15)
axes[1].barh(range(len(ret_imp)), ret_imp.values, color='#3498db')
axes[1].set_yticks(range(len(ret_imp)))
axes[1].set_yticklabels(ret_imp.index, fontsize=9)
axes[1].set_title('LightGBM Feature Importance (Return)', fontweight='bold')
axes[1].set_xlabel('Importance (Split Count)')
axes[1].invert_yaxis()

# LightGBM Volatility importance
vol_imp = pd.Series(lgb_vol_model.feature_importances_, index=feature_cols)
vol_imp = vol_imp.nlargest(15)
axes[2].barh(range(len(vol_imp)), vol_imp.values, color='#9b59b6')
axes[2].set_yticks(range(len(vol_imp)))
axes[2].set_yticklabels(vol_imp.index, fontsize=9)
axes[2].set_title('LightGBM Feature Importance (Volatility)', fontweight='bold')
axes[2].set_xlabel('Importance (Split Count)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Prediction Plots

In [ ]:
# ============================================================
# RETURN PREDICTIONS: ACTUAL vs PREDICTED
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- Time Series of Return Predictions (Test Set) ---
# Subsample for cleaner plot
plot_n = min(2000, len(dates_test))
plot_dates = dates_test[:plot_n]

axes[0, 0].plot(plot_dates, y_ret_test[:plot_n], alpha=0.5, linewidth=0.8, label='Actual', color='black')
axes[0, 0].plot(plot_dates, enet_pred_test[:plot_n], alpha=0.7, linewidth=0.8, label='ElasticNet', color='#e74c3c')
axes[0, 0].plot(plot_dates, lgb_ret_pred_test[:plot_n], alpha=0.7, linewidth=0.8, label='LightGBM', color='#3498db')
axes[0, 0].set_title(f'Return Prediction (Test Set, first {plot_n} bars)', fontweight='bold')
axes[0, 0].set_ylabel(f'{FORWARD_PERIOD}-Bar Forward Return')
axes[0, 0].legend(loc='upper right')
axes[0, 0].tick_params(axis='x', rotation=30)

# --- Scatter: ElasticNet ---
axes[0, 1].scatter(y_ret_test, enet_pred_test, alpha=0.15, s=5, color='#e74c3c')
lims = [min(y_ret_test.min(), enet_pred_test.min()), max(y_ret_test.max(), enet_pred_test.max())]
axes[0, 1].plot(lims, lims, 'k--', alpha=0.5, linewidth=1)
axes[0, 1].set_xlabel('Actual Return')
axes[0, 1].set_ylabel('Predicted Return')
axes[0, 1].set_title(f'ElasticNet: Actual vs Predicted (R²={r2_score(y_ret_test, enet_pred_test):.4f})', fontweight='bold')

# --- Scatter: LightGBM Return ---
axes[1, 0].scatter(y_ret_test, lgb_ret_pred_test, alpha=0.15, s=5, color='#3498db')
lims = [min(y_ret_test.min(), lgb_ret_pred_test.min()), max(y_ret_test.max(), lgb_ret_pred_test.max())]
axes[1, 0].plot(lims, lims, 'k--', alpha=0.5, linewidth=1)
axes[1, 0].set_xlabel('Actual Return')
axes[1, 0].set_ylabel('Predicted Return')
axes[1, 0].set_title(f'LightGBM: Actual vs Predicted Return (R²={r2_score(y_ret_test, lgb_ret_pred_test):.4f})', fontweight='bold')

# --- Scatter: LightGBM Volatility ---
axes[1, 1].scatter(y_vol_test, lgb_vol_pred_test, alpha=0.15, s=5, color='#9b59b6')
lims = [min(y_vol_test.min(), lgb_vol_pred_test.min()), max(y_vol_test.max(), lgb_vol_pred_test.max())]
axes[1, 1].plot(lims, lims, 'k--', alpha=0.5, linewidth=1)
axes[1, 1].set_xlabel('Actual Volatility')
axes[1, 1].set_ylabel('Predicted Volatility')
axes[1, 1].set_title(f'LightGBM: Actual vs Predicted Volatility (R²={r2_score(y_vol_test, lgb_vol_pred_test):.4f})', fontweight='bold')

plt.tight_layout()
plt.savefig('prediction_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VOLATILITY PREDICTION TIME SERIES
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(plot_dates, y_vol_test[:plot_n], alpha=0.6, linewidth=0.8, label='Actual Volatility', color='black')
ax.plot(plot_dates, lgb_vol_pred_test[:plot_n], alpha=0.7, linewidth=0.8, label='Predicted Volatility (LightGBM)', color='#9b59b6')
ax.fill_between(plot_dates, 0, y_vol_test[:plot_n], alpha=0.1, color='black')
ax.set_title(f'Volatility Prediction (Test Set, first {plot_n} bars)', fontweight='bold', fontsize=13)
ax.set_ylabel(f'{FORWARD_PERIOD}-Bar Forward Volatility')
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('volatility_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Trading Strategy: Signal Generation, Entry/Exit Rules & Position Sizing

**Strategy Logic:**
1. **Ensemble Signal**: Average of ElasticNet and LightGBM return predictions
2. **Entry**: Go long if predicted return > threshold; go short if < -threshold
3. **Exit**: Reverse position when signal flips, or flat if within dead zone
4. **Position Sizing**: Inverse-volatility × return-confidence (Kelly-inspired)

In [ ]:
# ============================================================
# STRATEGY IMPLEMENTATION
# ============================================================

# Build signals DataFrame on test set
signals = pd.DataFrame(index=dates_test)
signals['Actual_Return'] = y_ret_test
signals['Actual_Volatility'] = y_vol_test
signals['Pred_Return_ENet'] = enet_pred_test
signals['Pred_Return_LGB'] = lgb_ret_pred_test
signals['Pred_Volatility'] = lgb_vol_pred_test
signals['Close'] = df['Close'].iloc[split_idx:].values

# Ensemble predicted return (equal weight)
signals['Pred_Return_Ensemble'] = 0.5 * signals['Pred_Return_ENet'] + 0.5 * signals['Pred_Return_LGB']

# ---- ENTRY/EXIT RULES ----
# Threshold: enter when |predicted return| exceeds a multiple of predicted vol
ENTRY_THRESHOLD_MULT = 0.5  # Enter when pred_return > 0.5 * pred_vol
MAX_POSITION = 1.0          # Max position size (fraction of capital)

entry_threshold = ENTRY_THRESHOLD_MULT * signals['Pred_Volatility'].abs()

# Position direction: +1 long, -1 short, 0 flat
signals['Direction'] = 0
signals.loc[signals['Pred_Return_Ensemble'] > entry_threshold, 'Direction'] = 1
signals.loc[signals['Pred_Return_Ensemble'] < -entry_threshold, 'Direction'] = -1

# ---- POSITION SIZING ----
# Kelly-inspired: size proportional to |pred_return| / pred_vol²
# Clipped to [0, MAX_POSITION]
pred_vol_safe = signals['Pred_Volatility'].clip(lower=1e-6)
raw_size = signals['Pred_Return_Ensemble'].abs() / (pred_vol_safe ** 2)

# Normalize to [0, 1] range using rolling percentile
raw_size_rank = raw_size.rolling(window=500, min_periods=50).apply(
    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
)
signals['Position_Size'] = (raw_size_rank * MAX_POSITION).clip(0, MAX_POSITION).fillna(0.1)

# Final position = direction × size
signals['Position'] = signals['Direction'] * signals['Position_Size']

# ---- P&L CALCULATION ----
# Use 1-bar simple returns for P&L (the actual return over next bar)
signals['Next_1Bar_Return'] = df['Close'].iloc[split_idx:].pct_change().shift(-1).values
signals['Strategy_Return'] = signals['Position'] * signals['Next_1Bar_Return']
signals['Cumulative_Strategy'] = (1 + signals['Strategy_Return'].fillna(0)).cumprod()
signals['Cumulative_BuyHold'] = (1 + signals['Next_1Bar_Return'].fillna(0)).cumprod()

print(f"Signal distribution:")
print(signals['Direction'].value_counts().to_string())
print(f"\nAverage position size: {signals['Position_Size'].mean():.3f}")
print(f"Average absolute position: {signals['Position'].abs().mean():.3f}")

In [ ]:
# ============================================================
# STRATEGY PERFORMANCE METRICS
# ============================================================

def compute_strategy_metrics(returns, name='Strategy', periods_per_year=390*252):
    """Compute standard performance metrics for a return series."""
    returns = returns.dropna()
    total_return = (1 + returns).prod() - 1
    ann_return = (1 + total_return) ** (periods_per_year / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(periods_per_year)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Max drawdown
    cum = (1 + returns).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak
    max_dd = drawdown.min()
    
    # Win rate
    win_rate = (returns > 0).sum() / (returns != 0).sum() if (returns != 0).sum() > 0 else 0
    
    # Profit factor
    gross_profit = returns[returns > 0].sum()
    gross_loss = abs(returns[returns < 0].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    
    print(f"\n{'='*50}")
    print(f"{name} Performance Metrics")
    print(f"{'='*50}")
    print(f"Total Return:      {total_return*100:>10.2f}%")
    print(f"Annualized Return: {ann_return*100:>10.2f}%")
    print(f"Annualized Vol:    {ann_vol*100:>10.2f}%")
    print(f"Sharpe Ratio:      {sharpe:>10.3f}")
    print(f"Max Drawdown:      {max_dd*100:>10.2f}%")
    print(f"Win Rate:          {win_rate*100:>10.2f}%")
    print(f"Profit Factor:     {profit_factor:>10.3f}")
    print(f"Num Trades (bars): {(returns != 0).sum():>10d}")
    
    return {
        'total_return': total_return, 'ann_return': ann_return,
        'ann_vol': ann_vol, 'sharpe': sharpe, 'max_dd': max_dd,
        'win_rate': win_rate, 'profit_factor': profit_factor
    }

strat_metrics = compute_strategy_metrics(signals['Strategy_Return'], 'ML Strategy')
bh_metrics = compute_strategy_metrics(signals['Next_1Bar_Return'], 'Buy & Hold')

In [ ]:
# ============================================================
# CUMULATIVE RETURNS & DRAWDOWN PLOT
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 14), gridspec_kw={'height_ratios': [3, 1.5, 1.5]})

# Cumulative returns
axes[0].plot(signals.index, signals['Cumulative_Strategy'], label='ML Strategy', 
             color='#2ecc71', linewidth=1.5)
axes[0].plot(signals.index, signals['Cumulative_BuyHold'], label='Buy & Hold', 
             color='#3498db', linewidth=1.5, alpha=0.7)
axes[0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Cumulative Returns: ML Strategy vs Buy & Hold', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Growth of $1')
axes[0].legend(loc='upper left', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Drawdown
cum_strat = signals['Cumulative_Strategy']
peak_strat = cum_strat.cummax()
dd_strat = (cum_strat - peak_strat) / peak_strat

cum_bh = signals['Cumulative_BuyHold']
peak_bh = cum_bh.cummax()
dd_bh = (cum_bh - peak_bh) / peak_bh

axes[1].fill_between(signals.index, dd_strat, 0, alpha=0.4, color='#2ecc71', label='Strategy DD')
axes[1].fill_between(signals.index, dd_bh, 0, alpha=0.3, color='#3498db', label='Buy & Hold DD')
axes[1].set_title('Drawdown', fontweight='bold')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

# Position over time
axes[2].fill_between(signals.index, signals['Position'], 0, 
                      where=signals['Position'] > 0, color='#2ecc71', alpha=0.5, label='Long')
axes[2].fill_between(signals.index, signals['Position'], 0, 
                      where=signals['Position'] < 0, color='#e74c3c', alpha=0.5, label='Short')
axes[2].set_title('Position Size Over Time', fontweight='bold')
axes[2].set_ylabel('Position (fraction)')
axes[2].set_xlabel('Date')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# ROLLING SHARPE RATIO
# ============================================================

rolling_window = 5000  # ~13 trading days at minute freq
periods_per_year = 390 * 252

rolling_ret = signals['Strategy_Return'].rolling(rolling_window)
rolling_sharpe = (rolling_ret.mean() / rolling_ret.std()) * np.sqrt(periods_per_year)

rolling_bh_ret = signals['Next_1Bar_Return'].rolling(rolling_window)
rolling_sharpe_bh = (rolling_bh_ret.mean() / rolling_bh_ret.std()) * np.sqrt(periods_per_year)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(signals.index, rolling_sharpe, label='ML Strategy', color='#2ecc71', linewidth=1)
ax.plot(signals.index, rolling_sharpe_bh, label='Buy & Hold', color='#3498db', linewidth=1, alpha=0.7)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title(f'Rolling Sharpe Ratio ({rolling_window}-bar window)', fontweight='bold', fontsize=13)
ax.set_ylabel('Annualized Sharpe')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rolling_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison Summary

In [ ]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

summary = pd.DataFrame({
    'Model': ['ElasticNet (Return)', 'LightGBM (Return)', 'LightGBM (Volatility)'],
    'Train RMSE': [
        np.sqrt(mean_squared_error(y_ret_train, enet_pred_train)),
        np.sqrt(mean_squared_error(y_ret_train, lgb_ret_pred_train)),
        np.sqrt(mean_squared_error(y_vol_train, lgb_vol_pred_train))
    ],
    'Test RMSE': [
        np.sqrt(mean_squared_error(y_ret_test, enet_pred_test)),
        np.sqrt(mean_squared_error(y_ret_test, lgb_ret_pred_test)),
        np.sqrt(mean_squared_error(y_vol_test, lgb_vol_pred_test))
    ],
    'Train R²': [
        r2_score(y_ret_train, enet_pred_train),
        r2_score(y_ret_train, lgb_ret_pred_train),
        r2_score(y_vol_train, lgb_vol_pred_train)
    ],
    'Test R²': [
        r2_score(y_ret_test, enet_pred_test),
        r2_score(y_ret_test, lgb_ret_pred_test),
        r2_score(y_vol_test, lgb_vol_pred_test)
    ]
})

print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)
print(summary.to_string(index=False, float_format='{:.6f}'.format))

print("\n" + "="*70)
print("STRATEGY RULES SUMMARY")
print("="*70)
print(f"Forward prediction horizon: {FORWARD_PERIOD} bars")
print(f"Entry threshold: |pred_return| > {ENTRY_THRESHOLD_MULT} × pred_volatility")
print(f"Position sizing: Kelly-inspired inverse-volatility, max={MAX_POSITION}")
print(f"Direction: Long if pred > threshold, Short if pred < -threshold, Flat otherwise")
print(f"\nTest period: {dates_test[0].date()} to {dates_test[-1].date()}")
print(f"Strategy Sharpe: {strat_metrics['sharpe']:.3f}")
print(f"Buy&Hold Sharpe: {bh_metrics['sharpe']:.3f}")